In [ ]:
import socket

print(socket.gethostname())
import sys

sys.path.insert(0, "/home/hdashti/projects/shiplog/source")
from utils import load_raw, clean, dedup, summary, save_parquet


In [ ]:
import sys

sys.path.insert(0, "/home/hdashti/projects/shiplog/source")

# Force reload in case utils was already imported
import importlib, utils

importlib.reload(utils)
from utils import load_raw, clean, dedup, summary, save_parquet

df_raw = load_raw()
save_parquet(df_raw, filename="merged_raw.parquet")

df = clean(df_raw)
df = dedup(df)
save_parquet(df, filename="oldweather_cleaned_dedup.parquet")
summary(df)

Loaded: 354 files, 2,044,628 rows, 42 ships
Saved: /project/rcc/users/hdashti/projects/shiplogs/oldweather/oldweather_clean/merged_raw.parquet (33.5 MB)
Datetime: 2,044,606 valid, 22 missing
Lat range:  -78.5 to 82.7
Lon range:  -180.0 to 180.0
Baro range: 27.62 to 31.52
Suspect air temps: 1
Suspect coord jumps: 463
Before: 2,044,628 rows
After:  2,044,131 rows
Dropped: 497 rows

Rows dropped per ship:
ship
Burton_Island    76
Omaha            68
Tuscarora        49
Bear             48
Haida            47
Storis           42
Thetis           25
Lackawanna       25
Kearsarge        24
McCulloch        24
Onondaga         19
Chelan           19
Northwind        14
Atalanta          6
Northland         5
Manning           3
Panay             1
Ashuelot          1
Shoshone          1
dtype: int64
Saved: /project/rcc/users/hdashti/projects/shiplogs/oldweather/oldweather_clean/oldweather_cleaned_dedup.parquet (49.4 MB)
  OLD WEATHER CLEANED DATASET SUMMARY
  Total rows:     2,044,131
  Ships

In [ ]:
import pandas as pd

df = pd.read_parquet(
    "/project/rcc/users/hdashti/projects/shiplogs/oldweather/oldweather_clean/oldweather_cleaned_dedup.parquet"
)

df["year"] = df["datetime"].dt.year

# Define variable groups — each maps to one or more columns
variable_groups = {
    "temp_air": ["temp_dry", "temp_wet"],
    "temp_water": ["temp_water"],
    "pressure": ["baro"],
    "wind": ["wind_kts", "wind_dir_true", "wind_dir_mag"],
    "clouds": ["clouds", "cloud_amount", "clear_sky"],
    "weather": ["weather"],
    "ice": ["ice_log", "ice_1", "ice_2", "ice_index", "ice_terms"],
    "people": ["people"],
    "fauna": ["flora_fauna", "animals"],
}

# Count observations per (ship, year, variable)
edges = []
for var, cols in variable_groups.items():
    has_obs = (
        df[cols]
        .apply(
            lambda c: c.replace("", pd.NA).notna() if c.dtype == object else c.notna()
        )
        .any(axis=1)
    )
    sub = df[has_obs].groupby(["ship", "year"]).size().reset_index(name="count")
    sub["variable"] = var
    edges.append(sub)

edge_df = pd.concat(edges, ignore_index=True)
print(f"Total edges: {len(edge_df):,}")
print(edge_df.head(10))
print(f"\nUnique ships:     {edge_df['ship'].nunique()}")
print(f"Unique years:     {edge_df['year'].nunique()}")
print(f"Unique variables: {edge_df['variable'].nunique()}")

Total edges: 2,073
       ship    year  count  variable
0      Adak  1943.0   8727  temp_air
1      Adak  1944.0   8728  temp_air
2  Ashuelot  1866.0   6410  temp_air
3  Ashuelot  1867.0   8760  temp_air
4  Ashuelot  1868.0   8760  temp_air
5  Ashuelot  1869.0   8751  temp_air
6  Ashuelot  1874.0   8760  temp_air
7  Ashuelot  1879.0   7008  temp_air
8  Atalanta  1939.0   3123  temp_air
9  Atalanta  1941.0   1202  temp_air

Unique ships:     40
Unique years:     97
Unique variables: 9


In [ ]:
import pandas as pd
from pyvis.network import Network

df = pd.read_parquet(
    "/project/rcc/users/hdashti/projects/shiplogs/oldweather/oldweather_clean/oldweather_cleaned_dedup.parquet"
)
df["year"] = df["datetime"].dt.year
df["decade"] = (df["year"] // 10 * 10).astype("Int64")

variable_groups = {
    "temp_air": ["temp_dry", "temp_wet"],
    "temp_water": ["temp_water"],
    "pressure": ["baro"],
    "wind": ["wind_kts", "wind_dir_true", "wind_dir_mag"],
    "clouds": ["clouds", "cloud_amount", "clear_sky"],
    "weather": ["weather"],
    "ice": ["ice_log", "ice_1", "ice_2", "ice_index", "ice_terms"],
    "people": ["people"],
    "fauna": ["flora_fauna", "animals"],
}

# Build edges: (ship, decade, variable) → count
edges = []
for var, cols in variable_groups.items():
    has_obs = (
        df[cols]
        .apply(
            lambda c: c.replace("", pd.NA).notna() if c.dtype == object else c.notna()
        )
        .any(axis=1)
    )
    sub = (
        df[has_obs]
        .dropna(subset=["decade"])
        .groupby(["ship", "decade"])
        .size()
        .reset_index(name="count")
    )
    sub["variable"] = var
    edges.append(sub)
edge_df = pd.concat(edges, ignore_index=True)
edge_df["decade"] = edge_df["decade"].astype(int)

# Color per variable
var_colors = {
    "temp_air": "#E24B4A",
    "temp_water": "#185FA5",
    "pressure": "#534AB7",
    "wind": "#1D9E75",
    "clouds": "#888780",
    "weather": "#0F6E56",
    "ice": "#85B7EB",
    "people": "#D85A30",
    "fauna": "#639922",
}

# Build the network
net = Network(height="800px", width="100%", bgcolor="#ffffff", font_color="#222")
net.barnes_hut(gravity=-3000, central_gravity=0.3, spring_length=150)

# Decade nodes (top, blue)
for decade in sorted(edge_df["decade"].unique()):
    net.add_node(
        f"y_{decade}",
        label=f"{decade}s",
        color="#0C447C",
        shape="box",
        size=30,
        group="decade",
    )

# Variable nodes (middle, colored by type)
for var in variable_groups:
    if var in edge_df["variable"].unique():
        net.add_node(
            f"v_{var}",
            label=var,
            color=var_colors[var],
            shape="ellipse",
            size=25,
            group="variable",
        )

# Ship nodes (bottom, gray)
for ship in sorted(edge_df["ship"].unique()):
    net.add_node(
        f"s_{ship}", label=ship, color="#B4B2A9", shape="dot", size=15, group="ship"
    )

# Edges: decade ↔ variable (aggregated) and variable ↔ ship (aggregated)
dv = edge_df.groupby(["decade", "variable"])["count"].sum().reset_index()
for _, r in dv.iterrows():
    net.add_edge(
        f"y_{r['decade']}",
        f"v_{r['variable']}",
        value=int(r["count"]),
        title=f"{r['count']:,} obs",
    )

vs = edge_df.groupby(["variable", "ship"])["count"].sum().reset_index()
for _, r in vs.iterrows():
    net.add_edge(
        f"v_{r['variable']}",
        f"s_{r['ship']}",
        value=int(r["count"]),
        title=f"{r['count']:,} obs",
    )

# Save
out = "/home/hdashti/projects/shiplog/source/knowledge_graph.html"
net.write_html(out, notebook=False)
print(f"Saved: {out}")
print(f"Nodes: {len(net.nodes)}, Edges: {len(net.edges)}")

Saved: /home/hdashti/projects/shiplog/source/knowledge_graph.html
Nodes: 60, Edges: 369


In [ ]:
import pandas as pd

df = pd.read_parquet(
    "/project/rcc/users/hdashti/projects/shiplogs/oldweather/oldweather_clean/oldweather_cleaned_dedup.parquet"
)
df["year"] = df["datetime"].dt.year
print("Year range:", df["year"].min(), "to", df["year"].max())
print("Years with data:", sorted(df["year"].dropna().unique()))

Year range: 1859.0 to 1955.0
Years with data: [np.float64(1859.0), np.float64(1860.0), np.float64(1861.0), np.float64(1862.0), np.float64(1863.0), np.float64(1864.0), np.float64(1865.0), np.float64(1866.0), np.float64(1867.0), np.float64(1868.0), np.float64(1869.0), np.float64(1870.0), np.float64(1871.0), np.float64(1872.0), np.float64(1873.0), np.float64(1874.0), np.float64(1875.0), np.float64(1876.0), np.float64(1877.0), np.float64(1878.0), np.float64(1879.0), np.float64(1880.0), np.float64(1881.0), np.float64(1882.0), np.float64(1883.0), np.float64(1884.0), np.float64(1885.0), np.float64(1886.0), np.float64(1887.0), np.float64(1888.0), np.float64(1889.0), np.float64(1890.0), np.float64(1891.0), np.float64(1892.0), np.float64(1893.0), np.float64(1894.0), np.float64(1895.0), np.float64(1896.0), np.float64(1897.0), np.float64(1898.0), np.float64(1899.0), np.float64(1900.0), np.float64(1901.0), np.float64(1902.0), np.float64(1903.0), np.float64(1904.0), np.float64(1905.0), np.float64(19

In [ ]:
import pandas as pd
import plotly.graph_objects as go

df = pd.read_parquet(
    "/project/rcc/users/hdashti/projects/shiplogs/oldweather/oldweather_clean/oldweather_cleaned_dedup.parquet"
)
df["year"] = df["datetime"].dt.year
df["decade"] = (df["year"] // 10 * 10).astype("Int64")

variable_groups = {
    "temp_air": ["temp_dry", "temp_wet"],
    "temp_water": ["temp_water"],
    "pressure": ["baro"],
    "wind": ["wind_kts", "wind_dir_true", "wind_dir_mag"],
    "clouds": ["clouds", "cloud_amount", "clear_sky"],
    "weather": ["weather"],
    "ice": ["ice_log", "ice_1", "ice_2", "ice_index", "ice_terms"],
    "people": ["people"],
    "fauna": ["flora_fauna", "animals"],
}

# Build edge table
edges = []
for var, cols in variable_groups.items():
    has_obs = (
        df[cols]
        .apply(
            lambda c: c.replace("", pd.NA).notna() if c.dtype == object else c.notna()
        )
        .any(axis=1)
    )
    sub = (
        df[has_obs]
        .dropna(subset=["decade"])
        .groupby(["ship", "decade"])
        .size()
        .reset_index(name="count")
    )
    sub["variable"] = var
    edges.append(sub)
edge_df = pd.concat(edges, ignore_index=True)
edge_df["decade"] = edge_df["decade"].astype(int)

# Layout: three rows
decades = sorted(edge_df["decade"].unique())
variables = list(variable_groups.keys())
ships = sorted(edge_df["ship"].unique())


# x-coordinates spread evenly across each row (0..1)
def spread(items, y):
    n = len(items)
    return {item: (i / (n - 1) if n > 1 else 0.5, y) for i, item in enumerate(items)}


decade_pos = spread(decades, y=2)  # top row
var_pos = spread(variables, y=1)  # middle row
ship_pos = spread(ships, y=0)  # bottom row

# Edge color per variable
var_colors = {
    "temp_air": "#E24B4A",
    "temp_water": "#185FA5",
    "pressure": "#534AB7",
    "wind": "#1D9E75",
    "clouds": "#888780",
    "weather": "#0F6E56",
    "ice": "#85B7EB",
    "people": "#D85A30",
    "fauna": "#639922",
}

# Aggregate edges
dv = edge_df.groupby(["decade", "variable"])["count"].sum().reset_index()
vs = edge_df.groupby(["variable", "ship"])["count"].sum().reset_index()

fig = go.Figure()

# Decade ↔ variable edges
for _, r in dv.iterrows():
    x0, y0 = decade_pos[r["decade"]]
    x1, y1 = var_pos[r["variable"]]
    fig.add_trace(
        go.Scatter(
            x=[x0, x1],
            y=[y0, y1],
            mode="lines",
            line=dict(
                width=0.5 + r["count"] / dv["count"].max() * 4,
                color=var_colors[r["variable"]],
            ),
            opacity=0.4,
            hoverinfo="skip",
            showlegend=False,
        )
    )

# Variable ↔ ship edges
for _, r in vs.iterrows():
    x0, y0 = var_pos[r["variable"]]
    x1, y1 = ship_pos[r["ship"]]
    fig.add_trace(
        go.Scatter(
            x=[x0, x1],
            y=[y0, y1],
            mode="lines",
            line=dict(
                width=0.3 + r["count"] / vs["count"].max() * 3,
                color=var_colors[r["variable"]],
            ),
            opacity=0.3,
            hoverinfo="skip",
            showlegend=False,
        )
    )

# Decade nodes
fig.add_trace(
    go.Scatter(
        x=[decade_pos[d][0] for d in decades],
        y=[decade_pos[d][1] for d in decades],
        mode="markers+text",
        marker=dict(size=40, color="#0C447C", symbol="square"),
        text=[f"{d}s" for d in decades],
        textposition="top center",
        textfont=dict(size=12, color="#0C447C"),
        hovertext=[f"{d}s" for d in decades],
        hoverinfo="text",
        showlegend=False,
    )
)

# Variable nodes
fig.add_trace(
    go.Scatter(
        x=[var_pos[v][0] for v in variables],
        y=[var_pos[v][1] for v in variables],
        mode="markers+text",
        marker=dict(size=35, color=[var_colors[v] for v in variables]),
        text=variables,
        textposition="middle right",
        textfont=dict(size=11),
        hovertext=variables,
        hoverinfo="text",
        showlegend=False,
    )
)

# Ship nodes
fig.add_trace(
    go.Scatter(
        x=[ship_pos[s][0] for s in ships],
        y=[ship_pos[s][1] for s in ships],
        mode="markers+text",
        marker=dict(size=15, color="#888780"),
        text=ships,
        textposition="bottom center",
        textfont=dict(size=8, color="#444441"),
        hovertext=ships,
        hoverinfo="text",
        showlegend=False,
    )
)

fig.update_layout(
    title="Old Weather observations: decades → variables → ships",
    showlegend=False,
    hovermode="closest",
    xaxis=dict(visible=False, range=[-0.05, 1.05]),
    yaxis=dict(visible=False, range=[-0.3, 2.3]),
    plot_bgcolor="white",
    height=700,
    width=1400,
    margin=dict(l=20, r=20, t=60, b=60),
)

out = "/home/hdashti/projects/shiplog/source/knowledge_graph_plotly.html"
fig.write_html(out)
print(f"Saved: {out}")


Saved: /home/hdashti/projects/shiplog/source/knowledge_graph_plotly.html


In [ ]:
import pandas as pd
from pyvis.network import Network

df = pd.read_parquet(
    "/project/rcc/users/hdashti/projects/shiplogs/oldweather/oldweather_clean/oldweather_cleaned_dedup.parquet"
)
df["year"] = df["datetime"].dt.year
df["decade"] = (df["year"] // 10 * 10).astype("Int64")

variable_groups = {
    "temp_air": ["temp_dry", "temp_wet"],
    "temp_water": ["temp_water"],
    "pressure": ["baro"],
    "wind": ["wind_kts", "wind_dir_true", "wind_dir_mag"],
    "clouds": ["clouds", "cloud_amount", "clear_sky"],
    "weather": ["weather"],
    "ice": ["ice_log", "ice_1", "ice_2", "ice_index", "ice_terms"],
    "people": ["people"],
    "fauna": ["flora_fauna", "animals"],
}

# Build edges
edges = []
for var, cols in variable_groups.items():
    has_obs = (
        df[cols]
        .apply(
            lambda c: c.replace("", pd.NA).notna() if c.dtype == object else c.notna()
        )
        .any(axis=1)
    )
    sub = (
        df[has_obs]
        .dropna(subset=["decade"])
        .groupby(["ship", "decade"])
        .size()
        .reset_index(name="count")
    )
    sub["variable"] = var
    edges.append(sub)
edge_df = pd.concat(edges, ignore_index=True)
edge_df["decade"] = edge_df["decade"].astype(int)

# Compute total observations per node — drives node size
ship_totals = edge_df.groupby("ship")["count"].sum()
decade_totals = edge_df.groupby("decade")["count"].sum()
var_totals = edge_df.groupby("variable")["count"].sum()


# Helper: scale a value to a node size range
def scale(val, vmin, vmax, smin, smax):
    if vmax == vmin:
        return (smin + smax) / 2
    return smin + (val - vmin) / (vmax - vmin) * (smax - smin)


var_colors = {
    "temp_air": "#E24B4A",
    "temp_water": "#185FA5",
    "pressure": "#534AB7",
    "wind": "#1D9E75",
    "clouds": "#888780",
    "weather": "#0F6E56",
    "ice": "#85B7EB",
    "people": "#D85A30",
    "fauna": "#639922",
}

net = Network(height="900px", width="100%", bgcolor="#1a2332", font_color="#ffffff")

# Tuned physics — repulsion model with strong separation
net.set_options("""
{
  "physics": {
    "enabled": true,
    "solver": "forceAtlas2Based",
    "forceAtlas2Based": {
      "gravitationalConstant": -120,
      "centralGravity": 0.005,
      "springLength": 200,
      "springConstant": 0.05,
      "damping": 0.6,
      "avoidOverlap": 1
    },
    "stabilization": { "iterations": 300 }
  },
  "nodes": {
    "borderWidth": 2,
    "font": { "size": 14, "color": "#ffffff", "strokeWidth": 3, "strokeColor": "#1a2332" }
  },
  "edges": {
    "smooth": { "type": "continuous" },
    "color": { "inherit": false }
  }
}
""")

# Decade nodes — boxes, sized by total obs
dmin, dmax = decade_totals.min(), decade_totals.max()
for d in sorted(edge_df["decade"].unique()):
    size = scale(decade_totals[d], dmin, dmax, 25, 60)
    net.add_node(
        f"y_{d}",
        label=f"{d}s",
        color="#4A7BB7",
        shape="box",
        size=size,
        title=f"{d}s — {decade_totals[d]:,} observations",
    )

# Variable nodes — diamonds, colored, sized by total obs
vmin, vmax = var_totals.min(), var_totals.max()
for v in variable_groups:
    if v in var_totals.index:
        size = scale(var_totals[v], vmin, vmax, 25, 55)
        net.add_node(
            f"v_{v}",
            label=v,
            color=var_colors[v],
            shape="diamond",
            size=size,
            title=f"{v} — {var_totals[v]:,} observations",
        )

# Ship nodes — circles, sized by total obs
smin, smax = ship_totals.min(), ship_totals.max()
for s in sorted(edge_df["ship"].unique()):
    size = scale(ship_totals[s], smin, smax, 15, 50)
    net.add_node(
        f"s_{s}",
        label=s,
        color="#5DCAA5",
        shape="dot",
        size=size,
        title=f"{s} — {ship_totals[s]:,} observations",
    )

# Edges — thickness by count, color by variable
dv = edge_df.groupby(["decade", "variable"])["count"].sum().reset_index()
vs = edge_df.groupby(["variable", "ship"])["count"].sum().reset_index()
max_count = max(dv["count"].max(), vs["count"].max())

for _, r in dv.iterrows():
    width = scale(r["count"], 0, max_count, 0.3, 4)
    net.add_edge(
        f"y_{r['decade']}",
        f"v_{r['variable']}",
        width=width,
        color=var_colors[r["variable"]],
        title=f"{r['count']:,} obs",
    )

for _, r in vs.iterrows():
    width = scale(r["count"], 0, max_count, 0.3, 4)
    net.add_edge(
        f"v_{r['variable']}",
        f"s_{r['ship']}",
        width=width,
        color=var_colors[r["variable"]],
        title=f"{r['count']:,} obs",
    )

out = "/home/hdashti/projects/shiplog/source/scratch/knowledge_graph_pyvis.html"
net.write_html(out, notebook=False)
print(f"Saved: {out}")
print(f"Nodes: {len(net.nodes)}, Edges: {len(net.edges)}")

Saved: /home/hdashti/projects/shiplog/source/scratch/knowledge_graph_pyvis.html
Nodes: 60, Edges: 369


In [ ]:
import pandas as pd
from pyvis.network import Network

df = pd.read_parquet(
    "/project/rcc/users/hdashti/projects/shiplogs/oldweather/oldweather_clean/oldweather_cleaned_dedup.parquet"
)
df["year"] = df["datetime"].dt.year
df["decade"] = (df["year"] // 10 * 10).astype("Int64")

variable_groups = {
    "temp_air": ["temp_dry", "temp_wet"],
    "temp_water": ["temp_water"],
    "pressure": ["baro"],
    "wind": ["wind_kts", "wind_dir_true", "wind_dir_mag"],
    "clouds": ["clouds", "cloud_amount", "clear_sky"],
    "weather": ["weather"],
    "ice": ["ice_log", "ice_1", "ice_2", "ice_index", "ice_terms"],
    "people": ["people"],
    "fauna": ["flora_fauna", "animals"],
}

edges = []
for var, cols in variable_groups.items():
    has_obs = (
        df[cols]
        .apply(
            lambda c: c.replace("", pd.NA).notna() if c.dtype == object else c.notna()
        )
        .any(axis=1)
    )
    sub = (
        df[has_obs]
        .dropna(subset=["decade"])
        .groupby(["ship", "decade"])
        .size()
        .reset_index(name="count")
    )
    sub["variable"] = var
    edges.append(sub)
edge_df = pd.concat(edges, ignore_index=True)
edge_df["decade"] = edge_df["decade"].astype(int)

ship_totals = edge_df.groupby("ship")["count"].sum()
decade_totals = edge_df.groupby("decade")["count"].sum()
var_totals = edge_df.groupby("variable")["count"].sum()


def scale(val, vmin, vmax, smin, smax):
    if vmax == vmin:
        return (smin + smax) / 2
    return smin + (val - vmin) / (vmax - vmin) * (smax - smin)


var_colors = {
    "temp_air": "#E24B4A",
    "temp_water": "#185FA5",
    "pressure": "#534AB7",
    "wind": "#1D9E75",
    "clouds": "#888780",
    "weather": "#0F6E56",
    "ice": "#85B7EB",
    "people": "#D85A30",
    "fauna": "#639922",
}

net = Network(height="900px", width="100%", bgcolor="#1a2332", font_color="#ffffff")

# Tuned physics + hover-highlight
net.set_options("""
{
  "physics": {
    "enabled": true,
    "solver": "forceAtlas2Based",
    "forceAtlas2Based": {
      "gravitationalConstant": -150,
      "centralGravity": 0.01,
      "springLength": 250,
      "springConstant": 0.04,
      "damping": 0.6,
      "avoidOverlap": 1
    },
    "stabilization": { "iterations": 400 }
  },
  "nodes": {
    "borderWidth": 2,
    "font": { "size": 14, "color": "#ffffff", "strokeWidth": 3, "strokeColor": "#1a2332" }
  },
  "edges": {
    "smooth": { "type": "continuous" },
    "color": { "inherit": false }
  },
  "interaction": {
    "hover": true,
    "hoverConnectedEdges": true,
    "selectConnectedEdges": true
  }
}
""")

# Decade nodes — PINNED across the top in chronological order
decades_sorted = sorted(edge_df["decade"].unique())
canvas_width = 2000  # wider than display so decades spread out nicely
n_decades = len(decades_sorted)
dmin, dmax = decade_totals.min(), decade_totals.max()
for i, d in enumerate(decades_sorted):
    x = -canvas_width / 2 + (i / (n_decades - 1)) * canvas_width
    size = scale(decade_totals[d], dmin, dmax, 25, 60)
    net.add_node(
        f"y_{d}",
        label=f"{d}s",
        color="#4A7BB7",
        shape="box",
        size=size,
        title=f"{d}s — {decade_totals[d]:,} observations",
        x=x,
        y=-600,
        fixed={"x": True, "y": True},
        physics=False,
    )

# Variable nodes — diamonds, free-floating
vmin, vmax = var_totals.min(), var_totals.max()
for v in variable_groups:
    if v in var_totals.index:
        size = scale(var_totals[v], vmin, vmax, 25, 55)
        net.add_node(
            f"v_{v}",
            label=v,
            color=var_colors[v],
            shape="diamond",
            size=size,
            title=f"{v} — {var_totals[v]:,} observations",
        )

# Ship nodes — circles, free-floating
smin, smax = ship_totals.min(), ship_totals.max()
for s in sorted(edge_df["ship"].unique()):
    size = scale(ship_totals[s], smin, smax, 12, 55)
    net.add_node(
        f"s_{s}",
        label=s,
        color="#5DCAA5",
        shape="dot",
        size=size,
        title=f"{s} — {ship_totals[s]:,} observations",
    )

# Edges
dv = edge_df.groupby(["decade", "variable"])["count"].sum().reset_index()
vs = edge_df.groupby(["variable", "ship"])["count"].sum().reset_index()
max_count = max(dv["count"].max(), vs["count"].max())

for _, r in dv.iterrows():
    width = scale(r["count"], 0, max_count, 0.3, 4)
    net.add_edge(
        f"y_{r['decade']}",
        f"v_{r['variable']}",
        width=width,
        color=var_colors[r["variable"]],
        title=f"{r['count']:,} obs",
    )

for _, r in vs.iterrows():
    width = scale(r["count"], 0, max_count, 0.3, 4)
    net.add_edge(
        f"v_{r['variable']}",
        f"s_{r['ship']}",
        width=width,
        color=var_colors[r["variable"]],
        title=f"{r['count']:,} obs",
    )

out = "/home/hdashti/projects/shiplog/source/scratch/knowledge_graph_pyvis_v2.html"
net.write_html(out, notebook=False)
print(f"Saved: {out}")

Saved: /home/hdashti/projects/shiplog/source/scratch/knowledge_graph_pyvis_v2.html
